# 02 — Fine-Tune ViTPose++ on Walker-Gait Data

This notebook is the interactive development loop. The identical logic
runs on SHARCNET via `scripts/train.py`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import yaml
from pathlib import Path
from torch.utils.data import DataLoader

## 1. Load configs

In [ ]:
def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

train_cfg = load_yaml("../configs/train.yaml")
data_cfg = load_yaml("../configs/data.yaml")
model_cfg = load_yaml("../configs/model.yaml")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 2. Load model with freeze strategy

In [ ]:
from walker_gait.models.vitpose_finetune import load_model_for_finetune

model = load_model_for_finetune(
    model_name=model_cfg["backbone"]["name"],
    freeze_backbone=model_cfg["freeze"]["backbone"],
    unfreeze_last_n_blocks=model_cfg["freeze"]["unfreeze_last_n_blocks"],
    device=device,
)

## 3. Create datasets and dataloaders

In [ ]:
from walker_gait.data import GaitKeypointDataset

input_size = tuple(model_cfg["backbone"]["input_size"])   # (256, 192)
heatmap_size = (input_size[1] // 4, input_size[0] // 4)  # (48, 64)

train_ds = GaitKeypointDataset(
    ann_file=data_cfg["paths"]["train_ann"],
    img_dir=data_cfg["paths"]["frames_dir"],
    input_size=input_size,
    heatmap_size=heatmap_size,
    sigma=model_cfg["heatmap"]["sigma"],
    augment=True,
    scale_range=tuple(train_cfg["augmentation"]["random_scale"]),
    rotation_range=train_cfg["augmentation"]["random_rotation"],
)

val_ds = GaitKeypointDataset(
    ann_file=data_cfg["paths"]["val_ann"],
    img_dir=data_cfg["paths"]["frames_dir"],
    input_size=input_size,
    heatmap_size=heatmap_size,
    sigma=model_cfg["heatmap"]["sigma"],
    augment=False,
)

print(f"Train: {len(train_ds)} samples, Val: {len(val_ds)} samples")

train_loader = DataLoader(
    train_ds,
    batch_size=train_cfg["training"]["batch_size"],
    shuffle=True,
    num_workers=train_cfg["training"]["num_workers"],
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=train_cfg["training"]["batch_size"],
    shuffle=False,
    num_workers=train_cfg["training"]["num_workers"],
    pin_memory=True,
)

## 4. Sanity check — inspect one batch

In [ ]:
import matplotlib.pyplot as plt

batch = next(iter(train_loader))
print(f"pixel_values:    {batch['pixel_values'].shape}")
print(f"target_heatmaps: {batch['target_heatmaps'].shape}")
print(f"visibility:      {batch['visibility'].shape}")

# Show first sample's image and heatmap sum
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
img = batch["pixel_values"][0].permute(1, 2, 0).numpy()
img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]  # undo normalize
img = img.clip(0, 1)
axes[0].imshow(img)
axes[0].set_title("Input crop")

hm_sum = batch["target_heatmaps"][0].sum(dim=0).numpy()
axes[1].imshow(hm_sum, cmap="hot")
axes[1].set_title("Target heatmaps (sum)")
plt.tight_layout()
plt.show()

## 5. Training loop

In [ ]:
from walker_gait.training import train_one_epoch, evaluate

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=train_cfg["optimizer"]["lr"],
    weight_decay=train_cfg["optimizer"]["weight_decay"],
)

total_epochs = train_cfg["training"]["epochs"]
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_epochs - train_cfg["scheduler"]["warmup_epochs"],
    eta_min=train_cfg["scheduler"]["min_lr"],
)

history = {"train_loss": [], "val_loss": [], "val_pck": []}
best_pck = 0.0
ckpt_dir = Path("../checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for epoch in range(total_epochs):
    # Warmup
    if epoch < train_cfg["scheduler"]["warmup_epochs"]:
        warmup_lr = train_cfg["optimizer"]["lr"] * (epoch + 1) / train_cfg["scheduler"]["warmup_epochs"]
        for pg in optimizer.param_groups:
            pg["lr"] = warmup_lr

    train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
    history["train_loss"].append(train_loss)

    if epoch >= train_cfg["scheduler"]["warmup_epochs"]:
        scheduler.step()

    # Validate
    if (epoch + 1) % train_cfg["training"]["val_interval"] == 0:
        val_metrics = evaluate(
            model, val_loader, heatmap_size, input_size,
            pck_threshold=train_cfg["evaluation"]["pck_threshold"],
            device=device,
        )
        history["val_loss"].append(val_metrics["loss"])
        history["val_pck"].append(val_metrics["pck"])

        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, "
              f"val_loss={val_metrics['loss']:.4f}, val_pck={val_metrics['pck']:.4f}")

        if val_metrics["pck"] > best_pck:
            best_pck = val_metrics["pck"]
            model.save_pretrained(str(ckpt_dir / "best"))
            print(f"  ✓ New best PCK: {best_pck:.4f}")
    else:
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}")

model.save_pretrained(str(ckpt_dir / "last"))
print(f"\nDone. Best PCK: {best_pck:.4f}")

## 6. Plot training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="Train Loss")
if history["val_loss"]:
    val_epochs = list(range(
        train_cfg["training"]["val_interval"] - 1,
        len(history["train_loss"]),
        train_cfg["training"]["val_interval"],
    ))
    ax1.plot(val_epochs, history["val_loss"], label="Val Loss", marker="o")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.set_title("Loss")

if history["val_pck"]:
    ax2.plot(val_epochs, history["val_pck"], marker="o", color="green")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("PCK")
    ax2.set_title(f"Validation PCK (best: {best_pck:.4f})")

plt.tight_layout()
plt.show()